# Adaptive Learning Platform - Phase 1

This notebook prototypes accessible text transformation and quiz generation before API wiring. Run the cells from top to bottom.

## 1. SETUP

Imports, environment configuration, the shared LLM wrapper, and sample input.

In [1]:
import json
from dotenv import load_dotenv
load_dotenv()
import re
from typing import Any, Dict, List
import os
import pdfplumber
import requests

try:
    from groq import Groq
except ImportError:
    Groq = None

GROQ_API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("GROQ_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")
GROQ_CLIENT = None

if GROQ_API_KEY and Groq is not None:
    GROQ_CLIENT = Groq(api_key=GROQ_API_KEY)
    if not GROQ_MODEL:
        available_models = {model.id for model in GROQ_CLIENT.models.list().data}
        preferred_models = (
            "llama-3.3-70b-versatile",
            "llama-3.1-8b-instant",
            "openai/gpt-oss-20b",
            "openai/gpt-oss-120b",
        )
        GROQ_MODEL = next(
            (model for model in preferred_models if model in available_models),
            next((model for model in available_models if "llama" in model or "gpt" in model), None),
        )
    if not GROQ_MODEL:
        raise RuntimeError("No text-generation model is available for this Groq project. Set GROQ_MODEL in .env.")


def call_llm(prompt: str) -> str:
    """Generate text with Groq, or use the local fallback when no key is configured."""
    if GROQ_API_KEY:
        if GROQ_CLIENT is None:
            raise ImportError("Install the groq package in this notebook before using GROQ_API_KEY.")
        request_options = {
            "model": GROQ_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2,
            "max_completion_tokens": 4096,
        }
        if "QUIZ_JSON" in prompt or "CHUNK_JSON" in prompt:
            request_options["response_format"] = {"type": "json_object"}
        if GROQ_MODEL.startswith("openai/"):
            request_options["reasoning_effort"] = "low"
        response = GROQ_CLIENT.chat.completions.create(**request_options)
        return response.choices[0].message.content or ""

    if "QUIZ_JSON" in prompt:
        return json.dumps({
            "question": "What do plants use photosynthesis to produce?",
            "options": ["Glucose", "Sound", "Salt", "Metal"],
            "answer": "Glucose",
            "explanation": "Photosynthesis produces glucose, which stores chemical energy.",
        })
    if "CHUNK_JSON" in prompt:
        sentences = re.split(r"(?<=[.!?])\s+", prompt.split("TEXT:", 1)[-1].strip())
        return json.dumps({"chunks": [" ".join(sentences[index:index + 3]) for index in range(0, len(sentences), 3)]})
    return (
        "Photosynthesis lets plants make food from sunlight. Chlorophyll captures light energy. "
        "Plants use water and carbon dioxide to produce glucose and release oxygen."
    )

print("Groq SDK available:", Groq is not None)
print("Groq API configured:", bool(GROQ_API_KEY))
print("Groq model:", GROQ_MODEL or "local fallback")

Groq SDK available: True
Groq API configured: True
Groq model: openai/gpt-oss-20b


## 2. PDF EXTRACTION

PDF text extraction uses `pdfplumber`, with a page-level fallback for image-only or otherwise empty pages.

In [2]:
def extract_text_and_tables_from_pdf(filepath: str) -> tuple[str, List[List[List[str]]]]:
    """Extract prose text and tables separately so table rows don't bleed into paragraphs.

    Table regions are detected first via find_tables(), then excluded from the
    text extraction pass using page.filter(), so extract_text() only sees prose.
    """
    pages_text: List[str] = []
    pages_tables: List[List[List[List[str]]]] = []

    try:
        with pdfplumber.open(filepath) as pdf:
            for page_number, page in enumerate(pdf.pages, start=1):
                found_tables = page.find_tables()
                table_bboxes = [t.bbox for t in found_tables]  # (x0, top, x1, bottom)

                def is_inside_a_table(obj, boxes=table_bboxes) -> bool:
                    for (tx0, ttop, tx1, tbottom) in boxes:
                        if obj["x0"] >= tx0 and obj["x1"] <= tx1 and obj["top"] >= ttop and obj["bottom"] <= tbottom:
                            return True
                    return False

                prose_only_page = page.filter(lambda obj: not is_inside_a_table(obj))
                page_text = (prose_only_page.extract_text() or "").strip()
                if not page_text:
                    page_text = f"[No extractable prose text found on page {page_number}; OCR may be required.]"
                pages_text.append(page_text)

                pages_tables.append([t.extract() for t in found_tables])
    except FileNotFoundError:
        raise FileNotFoundError(f"PDF file not found: {filepath}")
    except Exception as error:
        raise RuntimeError(f"Could not extract PDF content from {filepath}: {error}") from error

    return "\n\n".join(pages_text), pages_tables


PDF_PATH = "./sample_input.pdf"

# No fallback: if extraction fails, stop here rather than silently using other text.
SOURCE_TEXT, SOURCE_TABLES = extract_text_and_tables_from_pdf(PDF_PATH)

print("--- Extracted prose text ---")
print(SOURCE_TEXT)

print("\n--- Extracted tables ---")
for page_num, page_tables in enumerate(SOURCE_TABLES, start=1):
    for table_num, table in enumerate(page_tables, start=1):
        print(f"\nPage {page_num}, table {table_num}:")
        for row in table:
            print(row)

print("\nSource words:", len(SOURCE_TEXT.split()))
print("Tables found:", sum(len(t) for t in SOURCE_TABLES))

--- Extracted prose text ---
Chapter 7: Photosynthesis and Energy Flow
Grade 9 Biology · Unit 3: Plant Systems
7.1 What is Photosynthesis?
Photosynthesis is the biochemical process by which green plants, algae, and certain bacteria
convert light energy, typically from the sun, into chemical energy stored in glucose molecules.
This process is fundamental to almost all life on Earth, as it forms the base of most food chains
and is responsible for producing the oxygen that most organisms depend on for cellular
respiration. The overall chemical equation for photosynthesis can be summarized as carbon
dioxide plus water, in the presence of light energy, yielding glucose and oxygen.
The process takes place primarily in the chloroplasts of plant cells, specialized organelles that
contain a green pigment called chlorophyll. Chlorophyll is essential because it absorbs light most
efficiently in the blue and red wavelengths of the visible spectrum, while reflecting green light,
which is why most p

## 3. PROFILE-BASED TRANSFORMATION

Prompt constants are deliberately separate from the transformation function so they can be edited without changing control flow.

In [3]:
DYSLEXIA_PROMPT = """Rewrite the text below for a reader with dyslexia. Use short, simple sentences and clear wording. Preserve every fact, relationship, number, and cause-and-effect detail. Do not add facts. Return only the rewritten text.

TEXT:
{text}"""

COGNITIVE_LOAD_PROMPT = """Split the text below into an ordered JSON object with one key, chunks. The chunks value must be an array of digestible chunks. Each chunk must contain 2 to 4 complete sentences. Preserve all original content and facts, do not summarize, and do not add facts. Return only valid JSON. Include the marker CHUNK_JSON nowhere except in this instruction context.

TEXT:
{text}"""

def _parse_json_response(response: str) -> Any:
    """Remove optional Markdown fences and parse a JSON response."""
    cleaned = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", response.strip(), flags=re.IGNORECASE)
    return json.loads(cleaned)

def transform_text(text: str, profile: str) -> Dict[str, Any]:
    """Transform text according to an accessibility profile."""
    if profile == "low_vision":
        return {"profile": profile, "text": text, "formatting": {"font_size_multiplier": 1.5, "contrast_mode": "high"}}
    if profile == "dyslexia":
        rewritten = call_llm(DYSLEXIA_PROMPT.format(text=text))
        if not rewritten.strip():
            raise ValueError("Groq returned an empty dyslexia transformation.")
        return {"profile": profile, "text": rewritten.strip()}
    if profile == "cognitive_load":
        response = call_llm(COGNITIVE_LOAD_PROMPT.format(text=text))
        parsed = _parse_json_response(response)
        chunks = parsed.get("chunks") if isinstance(parsed, dict) else parsed
        if not isinstance(chunks, list) or not all(isinstance(chunk, str) and chunk.strip() for chunk in chunks):
            raise ValueError("Cognitive-load response must contain a JSON chunks array of strings.")
        return {"profile": profile, "chunks": chunks}
    raise ValueError("profile must be dyslexia, low_vision, or cognitive_load")

for profile in ("dyslexia", "low_vision", "cognitive_load"):
    print(f"\n--- {profile.upper()} ---")
    print(json.dumps(transform_text(SOURCE_TEXT, profile), indent=2, ensure_ascii=False))


--- DYSLEXIA ---
{
  "profile": "dyslexia",
  "text": "Chapter 7: Photosynthesis and Energy Flow  \nGrade 9 Biology · Unit 3: Plant Systems  \n\n7.1 What is Photosynthesis?  \nPhotosynthesis is a chemical process.  \nGreen plants, algae, and some bacteria do it.  \nThey use light from the sun.  \nThey turn that light into chemical energy.  \nThe energy is stored in glucose molecules.  \nThis process is very important.  \nIt is the base of most food chains.  \nIt also makes the oxygen that most living things need.  \nThe overall equation is:  \ncarbon dioxide + water + light → glucose + oxygen.  \n\nThe process happens mainly in chloroplasts.  \nChloroplasts are special parts of plant cells.  \nThey contain a green pigment called chlorophyll.  \nChlorophyll absorbs light best in the blue and red parts of the spectrum.  \nIt reflects green light.  \nThat is why plants look green.  \n\nInside the chloroplast, photosynthesis has two stages.  \nEach stage happens in a different place.  \nE

In [4]:
import ipywidgets as widgets
from IPython.display import display

PROFILE_LABELS = {
    "I have dyslexia": "dyslexia",
    "I find long or dense text difficult": "cognitive_load",
    "I need larger, high-contrast text": "low_vision",
}

profile_selector = widgets.Dropdown(
    options=list(PROFILE_LABELS),
    value="I have dyslexia",
    description="I am dealing with:",
    layout=widgets.Layout(width="550px"),
)
text_input = widgets.Textarea(
    value=SOURCE_TEXT,   # was: value=SAMPLE_TEXT
    description="Text:",
    layout=widgets.Layout(width="700px", height="180px"),
)
generate_button = widgets.Button(description="Generate accessible version", button_style="primary")
transformation_output = widgets.Output()

def render_transformation(result: Dict[str, Any]) -> None:
    """Display the generated result in a form suited to the selected profile."""
    with transformation_output:
        transformation_output.clear_output()
        if result["profile"] == "cognitive_load":
            print("Generated digestible chunks:\n")
            for number, chunk in enumerate(result["chunks"], start=1):
                print(f"{number}. {chunk}\n")
        else:
            print(result["text"])
            if result["profile"] == "low_vision":
                print("\nDisplay settings: larger text (1.5x), high contrast")

def generate_selected_transformation(_button: widgets.Button) -> None:
    """Generate output from the user's selected profile and text."""
    text = text_input.value.strip()
    with transformation_output:
        transformation_output.clear_output()
        if not text:
            print("Please enter some text before generating an accessible version.")
            return
        try:
            selected_profile = PROFILE_LABELS[profile_selector.value]
            render_transformation(transform_text(text, selected_profile))
        except (ValueError, json.JSONDecodeError) as error:
            print(f"Could not generate the transformation: {error}")

generate_button.on_click(generate_selected_transformation)
display(widgets.VBox([profile_selector, text_input, generate_button, transformation_output]))

## 4. QUIZ GENERATION

Each chunk gets one LLM call. The parser accepts plain JSON and JSON wrapped in Markdown code fences.

In [5]:
QUIZ_PROMPT = """Create one practice question based only on the chunk below. Adjust phrasing complexity to the learner profile: {profile}. Return valid JSON only with exactly these keys: question, options, answer, explanation. The options value must be an array of exactly 4 strings. The answer must exactly match one option. Do not use information outside the chunk. Include the marker QUIZ_JSON nowhere except in this instruction context.

CHUNK:
{chunk}"""

def generate_quiz(chunk: str, profile: str) -> Dict[str, Any]:
    """Generate and validate one profile-aware quiz question for a text chunk."""
    response = call_llm(QUIZ_PROMPT.format(chunk=chunk, profile=profile))
    quiz = _parse_json_response(response)
    required_keys = {"question", "options", "answer", "explanation"}
    if set(quiz) != required_keys or not isinstance(quiz["options"], list) or len(quiz["options"]) != 4:
        raise ValueError("Quiz response must contain question, four options, answer, and explanation.")
    if quiz["answer"] not in quiz["options"]:
        raise ValueError("Quiz answer must match one of the options.")
    return quiz

cognitive_chunks = transform_text(SOURCE_TEXT, "cognitive_load")["chunks"]
for index, chunk in enumerate(cognitive_chunks[:3], start=1):
    print(f"\n--- QUIZ FOR CHUNK {index} ---")
    print(json.dumps(generate_quiz(chunk, "cognitive_load"), indent=2))


--- QUIZ FOR CHUNK 1 ---


ValueError: Quiz response must contain question, four options, answer, and explanation.

## 5. END-TO-END TEST

This final cell runs raw text through all three profile paths and generates practice questions for the transformed content.

In [ ]:
def chunks_for_quiz(transformed: Dict[str, Any]) -> List[str]:
    """Normalize a transformed profile result into quiz-ready chunks."""
    if transformed["profile"] == "cognitive_load":
        return transformed["chunks"]
    return [transformed["text"]]

# Section 5 end-to-end test
pipeline_summary: Dict[str, Any] = {}
for profile in ("dyslexia", "low_vision", "cognitive_load"):
    transformed = transform_text(SOURCE_TEXT, profile)
    quiz_results = [generate_quiz(chunk, profile) for chunk in chunks_for_quiz(transformed)[:3]]
    pipeline_summary[profile] = {
        "transformed_keys": list(transformed.keys()),
        "quiz_count": len(quiz_results),
        "first_question": quiz_results[0]["question"] if quiz_results else None,
    }

print("\n=== FINAL PIPELINE SUMMARY ===")
print(json.dumps(pipeline_summary, indent=2))